# Fine-tune SVTR26-Tiny Chuyên Ô tô (4 ngày 02..05/08 Train/Val — Ngày 01/08 làm Test độc lập)

- **Trọng số khởi tạo:** Ưu tiên dùng **model đã fine-tune vòng 1 của bạn** (`/content/drive/MyDrive/LITEALPR_FINETUNE_VN/best.pth` — bản gốc sinh ra `best.onnx`, vì PyTorch cần file `.pth` để huấn luyện tiếp). Nếu không tìm thấy mới tự tải `svtr26_tiny/best.pth` từ Hugging Face.
- **Tập dữ liệu `tap_finetune_oto.zip`:**
  - `Train` & `Val`: Trích nhãn đối chiếu chéo từ **4 ngày `2026-08-02 .. 2026-08-05`** (đã loại bỏ sạch mọi xe từng xuất hiện ở ngày `01/08` để chống rò rỉ dữ liệu 100%).
  - `Test`: Toàn bộ ảnh Ô tô ngày **`2026-08-01`** bạn đã tự tay duyệt.
- **Đầu ra:** Gộp thành **1 file duy nhất `svtr26_oto.onnx`** lưu thẳng vào Google Drive.

In [ ]:
# Bước 1: Kết nối Google Drive và kiểm tra GPU
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi

In [ ]:
# Bước 2: Tải mã nguồn LiteALPR, cài thư viện và vá tương thích NumPy 2.0+
%cd /content
!rm -rf /content/LiteALPR
!git clone https://github.com/vn-anhnth/LiteALPR.git
%cd /content/LiteALPR
!pip install -q ultralytics imgaug pyclipper onnxscript onnxslim lmdb rapidfuzz onnx onnxruntime-gpu huggingface_hub pyyaml tqdm pillow opencv-python
# Vá tương thích imgaug 0.4.0 trên NumPy 2.0+ bằng find_spec (không import imgaug trước khi vá!)
import importlib.util, pathlib
p_imgaug = pathlib.Path(importlib.util.find_spec('imgaug').submodule_search_locations[0]) / 'imgaug.py'
noi_dung = p_imgaug.read_text()
ban_va = 'if not hasattr(np, "sctypes"): np.sctypes = {"int":[np.int8,np.int16,np.int32,np.int64],"uint":[np.uint8,np.uint16,np.uint32,np.uint64],"float":[np.float16,np.float32,np.float64],"complex":[np.complex64,np.complex128],"others":[bool,object,bytes,str,np.void]}\n'
if 'hasattr(np, "sctypes")' not in noi_dung:
    p_imgaug.write_text(noi_dung.replace('import numpy as np\n', 'import numpy as np\n' + ban_va, 1))
print('✅ Đã vá imgaug thành công!')

In [ ]:
# Bước 3: Giải nén tap_finetune_oto.zip (upload file tap_finetune_oto.zip mới lên MyDrive trước)
DUONG_DAN_ZIP = '/content/drive/MyDrive/tap_finetune_oto.zip'

!rm -rf /content/du_lieu_oto /content/lmdb_oto
!mkdir -p /content/du_lieu_oto
!unzip -q -o "{DUONG_DAN_ZIP}" -d /content/du_lieu_oto
!wc -l /content/du_lieu_oto/*_labels.txt

In [ ]:
# Bước 4: Đóng gói sang định dạng LMDB
%cd /content/LiteALPR
!python tools/create_lmdb_dataset.py \
  --data_dir /content/du_lieu_oto \
  --label_files train_labels.txt val_labels.txt test_labels.txt \
  --output_dir /content/lmdb_oto
!ls -lh /content/lmdb_oto

In [ ]:
# Bước 5: Chọn trọng số khởi tạo (ưu tiên best.pth vòng 1 của bạn trên Drive) & đo mốc trước khi train trên ngày 01/08
from pathlib import Path
import shutil
from huggingface_hub import hf_hub_download

PTH_VONG_1 = Path('/content/drive/MyDrive/LITEALPR_FINETUNE_VN/best.pth')
TRONG_SO_KHOI_TAO = '/content/pretrained_init.pth'

if PTH_VONG_1.exists():
    shutil.copy(PTH_VONG_1, TRONG_SO_KHOI_TAO)
    print('✅ Đang dùng trọng số Fine-tuned vòng 1 của bạn:', PTH_VONG_1)
else:
    duong_dan_goc = hf_hub_download(repo_id='anhone3/LiteALPR', filename='svtr26_tiny/best.pth')
    shutil.copy(duong_dan_goc, TRONG_SO_KHOI_TAO)
    print('⚠️ Không thấy LITEALPR_FINETUNE_VN/best.pth trên Drive -> Dùng svtr26_tiny/best.pth gốc')

%cd /content/LiteALPR
print('--- Độ chính xác TRƯỚC khi fine-tune thêm trên tập Test độc lập ngày 01/08 ---')
!python tools/eval_rec.py \
  -c configs/rec/svtr26/svtr26_tiny.yml \
  -o Global.pretrained_model={TRONG_SO_KHOI_TAO} \
     Eval.loader.batch_size_per_card=64 \
     Eval.sampler.first_bs=64 \
     Eval.dataset.data_dir_list="['/content/lmdb_oto/test']"

In [ ]:
# Bước 6: Fine-tune vòng 2 trên dữ liệu Ô tô 4 ngày (02..05/08)
THU_MUC_LUU_DRIVE = '/content/drive/MyDrive/LITEALPR_FINETUNE_OTO'
!mkdir -p "{THU_MUC_LUU_DRIVE}"

%cd /content/LiteALPR
!python tools/train_rec.py \
  -c configs/rec/svtr26/svtr26_tiny.yml \
  -o Global.pretrained_model=/content/pretrained_init.pth \
     Global.output_dir="{THU_MUC_LUU_DRIVE}" \
     Global.epoch_num=20 \
     Global.eval_epoch_step="[0, 1]" \
     LRScheduler.warmup_epoch=2 \
     Optimizer.lr=0.00015 \
     Train.loader.batch_size_per_card=64 \
     Train.sampler.first_bs=64 \
     Eval.loader.batch_size_per_card=64 \
     Eval.sampler.first_bs=64 \
     Train.dataset.data_dir_list="['/content/lmdb_oto/train']" \
     Eval.dataset.data_dir_list="['/content/lmdb_oto/val']"

In [ ]:
# Bước 7: Đánh giá SAU Fine-tune trên tập Test độc lập ngày 01/08 & Xuất ra 1 file duy nhất svtr26_oto.onnx
%cd /content/LiteALPR
print('--- Độ chính xác SAU khi fine-tune trên tập Test độc lập ngày 01/08 ---')
!python tools/eval_rec.py \
  -c configs/rec/svtr26/svtr26_tiny.yml \
  -o Global.pretrained_model="{THU_MUC_LUU_DRIVE}/best.pth" \
     Eval.loader.batch_size_per_card=64 \
     Eval.sampler.first_bs=64 \
     Eval.dataset.data_dir_list="['/content/lmdb_oto/test']"

!mkdir -p /content/onnx_tmp
!python tools/export_rec.py \
  -m "{THU_MUC_LUU_DRIVE}/best.pth" \
  --save_path /content/onnx_tmp/raw.onnx \
  --dynamic

import onnx
m = onnx.load('/content/onnx_tmp/raw.onnx', load_external_data=True)
duong_dan_onnx_cuoi = f'{THU_MUC_LUU_DRIVE}/svtr26_oto.onnx'
onnx.save_model(m, duong_dan_onnx_cuoi, save_as_external_data=False)
print('✅ ĐÃ XUẤT XONG FILE DUY NHẤT:', duong_dan_onnx_cuoi)
!ls -lh "{duong_dan_onnx_cuoi}"